# Training Demo - New Pipeline

This notebook demonstrates the new end-to-end training pipeline with:
- Proper data loading with augmentation
- Device detection (CUDA/MPS/CPU)
- Training with Mixup/CutMix
- Model checkpointing
- Complete reproducibility

**Note**: For full training, use the command-line scripts:
```bash
python scripts/train.py --model resnet
python scripts/finetune.py --model resnet --pretrained models/all_models/resnet_best.pth
```

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import torch
import logging
from src.config import load_config
from src.utils import print_device_info, get_device, count_parameters
from src.dataset import create_dataloaders, FASHION_MNIST_CLASSES
from src.model_definitions import ResNet, BasicBlock, MiniCNN, TinyVGG
from src.data_augmentation import AugmentationPipeline

logging.basicConfig(level=logging.INFO)
print("✅ All modules imported successfully!")

## 1. Device Detection

The new pipeline automatically detects the best available device:
- **CUDA** (NVIDIA GPU) - highest priority
- **MPS** (Apple Silicon) - for M1/M2 MacBooks
- **CPU** - fallback

In [ ]:
# Detect and print device information
device = print_device_info()

## 2. Load Configuration

All training parameters are now centralized in `config.yaml`:

In [ ]:
# Load configuration
config = load_config('../config.yaml')

print("Training Configuration:")
print(f"  Epochs: {config.training.epochs}")
print(f"  Batch Size: {config.training.batch_size}")
print(f"  Learning Rate: {config.training.learning_rate}")
print(f"  Optimizer: {config.training.optimizer}")
print(f"  Early Stopping Patience: {config.training.early_stopping_patience}")

print("\nAugmentation Configuration:")
print(f"  Enabled: {config.augmentation.enabled}")
print(f"  Mixup: {config.augmentation.mixup}")
print(f"  CutMix: {config.augmentation.cutmix}")
print(f"  Rotation: {config.augmentation.rotation}°")
print(f"  Horizontal Flip: {config.augmentation.horizontal_flip}")

## 3. Create Dataloaders with Augmentation

The new `create_dataloaders()` function handles everything:

In [ ]:
# Create dataloaders (using torchvision FashionMNIST)
train_loader, val_loader, test_loader = create_dataloaders(
    use_torchvision=True,
    batch_size=config.training.batch_size,
    num_workers=0
)

print(f"✅ Dataloaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

# Test a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels: {labels[:8].tolist()}")
print(f"Class names: {[FASHION_MNIST_CLASSES[i] for i in labels[:8]]}")

## 4. Setup Augmentation Pipeline

Configure augmentations based on config.yaml:

In [ ]:
# Setup augmentation pipeline
aug_config = {}

if config.augmentation.mixup:
    aug_config['mixup'] = {'alpha': config.augmentation.mixup_alpha}

if config.augmentation.cutmix:
    aug_config['cutmix'] = {'alpha': config.augmentation.cutmix_alpha}

# Add torchvision transforms
if config.augmentation.rotation > 0 or config.augmentation.horizontal_flip:
    aug_config['torchvision'] = {
        'rotation': config.augmentation.rotation,
        'horizontal_flip': config.augmentation.horizontal_flip,
        'vertical_flip': config.augmentation.vertical_flip
    }

augmentation_pipeline = AugmentationPipeline(aug_config)
print(f"✅ Augmentation pipeline created with: {list(aug_config.keys())}")

# Test Mixup
if augmentation_pipeline.mixup_enabled:
    print("\nTesting Mixup...")
    mixed_x, y_a, y_b, lam = augmentation_pipeline.apply_mixup(images[:8], labels[:8])
    print(f"   Mixed images shape: {mixed_x.shape}")
    print(f"   Lambda: {lam:.3f}")
    print(f"   Original labels: {labels[:8].tolist()}")
    print(f"   Mixed with labels: {y_b.tolist()}")

## 5. Create and Inspect Model

Create a model and check its parameters:

In [ ]:
# Create ResNet model
model = ResNet(BasicBlock, [2, 2, 2, 2], num_classes=10)
model = model.to(device)

print(f"✅ ResNet created")
print(f"   Total parameters: {count_parameters(model):,}")
print(f"   Device: {next(model.parameters()).device}")

# Test forward pass
model.eval()
with torch.no_grad():
    test_output = model(images[:4].to(device))
    print(f"\nTest forward pass:")
    print(f"   Input shape: {images[:4].shape}")
    print(f"   Output shape: {test_output.shape}")
    print(f"   Predictions: {torch.argmax(test_output, dim=1).cpu().tolist()}")

## 6. Quick Training Test (1 Epoch)

Test the training loop with augmentation:

In [ ]:
import torch.nn as nn
import torch.optim as optim
from src.utils import train_step, validation_step

# Setup training components
model.train()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("🚀 Running 1 epoch training test...\n")

# Train for 1 epoch (without Mixup/CutMix for simplicity in notebook)
train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer, device)
val_loss, val_acc = validation_step(model, val_loader, loss_fn, device)

print(f"Epoch 1 Results:")
print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
print(f"\n✅ Training pipeline works!")

## 7. Using the Command-Line Scripts

For full training, use the CLI scripts which include all features:

### Train from scratch:
```bash
# Train all models
python scripts/train.py --model all

# Train specific model
python scripts/train.py --model resnet --config config.yaml

# Train with CPU (force)
python scripts/train.py --model minicnn --force-cpu
```

### Fine-tune with hyperparameter search:
```bash
# Fine-tune ResNet
python scripts/finetune.py --model resnet \
    --pretrained models/all_models/resnet_best.pth \
    --learning-rates 1e-5 5e-6 \
    --batch-sizes 32 64 \
    --patience-values 2 3

# Quick test with fewer combinations
python scripts/finetune.py --model minicnn \
    --learning-rates 1e-4 \
    --batch-sizes 32 \
    --patience-values 2
```

### Prepare data (if using CSV format):
```bash
python scripts/prepare_data.py --output-dir data_preparation
```

## Key Improvements:

1. ✅ **Data augmentation actually works** - Mixup, CutMix, transforms integrated
2. ✅ **Device auto-detection** - CUDA, MPS (Apple Silicon), CPU
3. ✅ **Config-driven** - Everything in config.yaml
4. ✅ **Reproducible** - CLI scripts for consistent training
5. ✅ **Modular** - Clean separation: data, models, training, evaluation
6. ✅ **Checkpointing** - Saves best models automatically
7. ✅ **Early stopping** - Prevents overfitting
8. ✅ **Metrics tracking** - JSON history files

## Next Steps:

1. Run full training: `python scripts/train.py --model all`
2. Fine-tune best model: `python scripts/finetune.py --model resnet`
3. Evaluate: `python main.py --model_path models/all_models/resnet_best.pth --test_csv data/.../test.csv`

---
**🎉 The FashionMNIST-Analysis project now has a complete, production-ready pipeline!**